# Epinet

In [ ]:
%%capture
%pip install git+https://github.com/lightning-uq-box/lightning-uq-box.git

## Theoretic Foundation

An *epistemic neural network* (ENN), introduced by [Osband et al., 2023](https://arxiv.org/abs/2107.08924), is a pair: a function class $f_\theta(x, z)$ and a reference distribution $P_Z$ over an **epistemic index** $z$. Where a conventional network maps an input to a single prediction, an ENN maps an input *and an index* to a prediction, and integrating over $z \sim P_Z$ recovers a predictive distribution.

What this buys is *joint* predictions. Most UQ methods are evaluated one input at a time, on marginal predictions $\hat{P}(y_t \mid x_t)$. But many downstream uses — sequential decision making, active learning, bandits — depend on how predictions at *different* inputs covary, which the marginals cannot express. An ENN's joint prediction over $\tau$ inputs is

$$\hat{P}_{1:\tau}(y_{1:\tau}) = \int P_Z(dz) \prod_{t=1}^{\tau} \mathrm{softmax}(f_\theta(x_t, z))_{y_t},$$

where the shared $z$ inside the product is what ties the predictions together.

The **epinet** is a particular, cheap way of building one. Rather than a new architecture, it is a small head bolted onto a conventional base network $\mu_\zeta(x)$:

$$f_\theta(x, z) = \mu_\zeta(x) + \sigma_\eta(\mathrm{sg}[\phi_\zeta(x)], z),$$

where $\phi_\zeta(x)$ are features of the base network's last hidden layer and $\mathrm{sg}[\cdot]$ is a stop-gradient: the epinet reads the base network's features but never sends gradients back through them. The epinet itself splits into a learnable part and a *fixed* part,

$$\sigma_\eta(\tilde{x}, z) = \sigma^L_\eta(\tilde{x}, z) + \sigma^P(\tilde{x}, z), \qquad \sigma^L_\eta(\tilde{x}, z) := \mathrm{mlp}_\eta([\tilde{x}, z])^T z.$$

The learnable part is an MLP whose output is reshaped to $\mathbb{R}^{D_Z \times C}$ and contracted with the index. The fixed part $\sigma^P$ is a frozen, randomly initialized *prior function*. It is never trained, and it is what supplies uncertainty where the data does not pin the model down: away from the training data nothing pulls the learnable part toward cancelling it, so the prior's variation across $z$ survives as predictive spread.

Two properties make this practical. Because the base network is only read, never modified, an epinet can be attached to a **large pretrained network whose weights stay frozen** — the paper's headline result is an epinet beating a 100-particle deep ensemble on ImageNet joint log-loss at less than the cost of two particles. And because only the small head sees the index, averaging the loss over many index draws costs one base forward pass, not many.

## Imports

In [ ]:
import os
import tempfile
from functools import partial

import matplotlib.pyplot as plt
from lightning import Trainer
from lightning.pytorch import seed_everything
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.optim import Adam

from lightning_uq_box.datamodules import ToyHeteroscedasticDatamodule
from lightning_uq_box.models import MLP
from lightning_uq_box.uq_methods import EpinetRegression
from lightning_uq_box.viz_utils import (
    plot_calibration_uq_toolbox,
    plot_predictions_regression,
    plot_toy_regression_data,
    plot_training_metrics,
)

plt.rcParams["figure.figsize"] = [14, 5]

In [ ]:
seed_everything(0)  # seed everything for reproducibility

We define a temporary directory to look at some training metrics and results.

In [ ]:
my_temp_dir = tempfile.mkdtemp()

## Datamodule

To demonstrate the method, we will make use of a Toy Regression Example that is defined as a [Lightning Datamodule](https://lightning.ai/docs/pytorch/stable/data/datamodule.html). While this might seem like overkill for a small toy problem, we think it is more helpful how the individual pieces of the library fit together so you can train models on more complex tasks.

In [ ]:
dm = ToyHeteroscedasticDatamodule()

X_train, Y_train, train_loader, X_test, Y_test, test_loader, X_gtext, Y_gtext = (
    dm.X_train,
    dm.Y_train,
    dm.train_dataloader(),
    dm.X_test,
    dm.Y_test,
    dm.test_dataloader(),
    dm.X_gtext,
    dm.Y_gtext,
)

In [ ]:
fig = plot_toy_regression_data(X_train, Y_train, X_test, Y_test)

## Model

The epinet attaches to an existing network, so we start with an ordinary base network — here a simple Multi-layer Perceptron (MLP). Nothing about it is epinet-specific, and in a realistic setting this would be a large pretrained model. For the documentation of the MLP see [here](https://readthedocs.io/en/stable/api/models.html#MLP).

In [ ]:
network = MLP(n_inputs=1, n_hidden=[50, 50], n_outputs=1, activation_fn=nn.ReLU())
network

The `EpinetRegression` wrapper attaches the epinet head. It locates the base network's output layer, registers a forward pre-hook that captures the layer's input — those are the features $\phi_\zeta(x)$ — and builds the small index-consuming head around them. The base network itself is left completely untouched, which is why this also works on a frozen pretrained model.

The parameters worth knowing about:

- `index_dim` is $D_Z$, the dimension of the epistemic index. Larger means a richer family of joint predictions.
- `num_index_samples` is how many indices the training loss is averaged over per batch. The base network still runs only once; the batch is repeated for the cheap head.
- `input_prior_scale` scales the frozen prior function over the raw input. **This is the knob that produces epistemic uncertainty.** At `0.0` the credible band collapses to something flat and uninformative.
- `epi_prior_scale` scales the epinet's own frozen prior over the features. The Neural Testbed configuration leaves this at `0.0` and puts all prior variation in the input prior, which is what we do here.

In [ ]:
epinet_module = EpinetRegression(
    model=network,
    optimizer=partial(Adam, lr=1e-2),
    loss_fn=nn.MSELoss(),
    index_dim=8,
    num_index_samples=8,
    num_pred_samples=100,
    epinet_hidden_dims=[15, 15],
    prior_hidden_dims=[5, 5],
    epi_prior_scale=0.0,
    input_prior_scale=3.0,
)

## Trainer

Now that we have a LightningDataModule and a UQ-Method as a LightningModule, we can conduct training with a [Lightning Trainer](https://lightning.ai/docs/pytorch/stable/common/trainer.html). It has tons of options to make your life easier, so we encourage you to check the documentation.

In [ ]:
logger = CSVLogger(my_temp_dir)
trainer = Trainer(
    accelerator="cpu",
    max_epochs=250,  # number of epochs we want to train
    logger=logger,  # log training metrics for later evaluation
    log_every_n_steps=1,
    enable_checkpointing=False,
    enable_progress_bar=False,
    default_root_dir=my_temp_dir,
)

Training our model is now easy:

In [ ]:
trainer.fit(epinet_module, dm)

## Training Metrics

To get some insights into how the training went, we can use the utility function to plot the training loss and RMSE metric.

In [ ]:
fig = plot_training_metrics(
    os.path.join(my_temp_dir, "lightning_logs"), ["train_loss", "trainRMSE"]
)

## Prediction

For prediction we can either rely on the `trainer.test()` method or manually conduct a `predict_step()`. Using the trainer will save the predictions and some metrics to a CSV file, while the manual `predict_step()` with a single input tensor will generate a dictionary that holds the mean prediction as well as some other quantities of interest, for example the predicted standard deviation.

In [ ]:
# save predictions
trainer.test(epinet_module, dm.test_dataloader())

## Evaluate Predictions

The constructed Data Module contains two possible test variable. `X_test` are IID samples from the same noise distribution as the training data, while `X_gtext` ("X ground truth extended") are dense inputs from the underlying "ground truth" function without any noise that also extends the input range to either side, so we can visualize the method's UQ tendencies when extrapolating beyond the training data range. Thus, we will use `X_gtext` for visualization purposes, but use `X_test` to compute uncertainty and calibration metrics because we want to analyse how well the method has learned the noisy data distribution.

In [ ]:
preds = epinet_module.predict_step(X_gtext.to(epinet_module.device))

fig = plot_predictions_regression(
    X_train,
    Y_train,
    X_gtext,
    Y_gtext,
    preds["pred"],
    preds["pred_uct"],
    epistemic=preds["epistemic_uct"],
    title="Epinet",
)

The credible band should **widen away from the training data**, which is the behaviour the prior function is there to produce. Inside the training range the learnable part of the epinet has been pushed to cancel the prior's variation across indices, so the band is narrow; outside it, nothing has pulled it into agreement, and the prior's spread shows through.

This is also the most useful diagnostic if something is wired up wrong. A band that stays flat across the whole input range almost always means the prior function is contributing nothing — check that at least one of `input_prior_scale` and `epi_prior_scale` is non-zero.

Because `predict_step` also returns the raw ENN samples, we can look directly at the individual index draws rather than only their summary statistics. Each curve below is the network's prediction under one fixed $z$, and the way the curves fan out away from the data is the same story the credible band tells.

In [ ]:
samples = preds["samples"].squeeze(1).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(20):
    ax.plot(X_gtext.cpu().numpy(), samples[:, i], color="C0", alpha=0.3, lw=1)
ax.scatter(
    X_train.cpu().numpy(), Y_train.cpu().numpy(), s=8, color="C1", label="train data"
)
ax.set_title("Individual epinet index samples $f_\\theta(x, z_i)$")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
plt.show()

For some additional metrics relevant to UQ, we can use the great [uncertainty-toolbox](https://uncertainty-toolbox.github.io/) that gives us some insight into the calibration of our prediction.

In [ ]:
preds = epinet_module.predict_step(X_test.to(epinet_module.device))
fig = plot_calibration_uq_toolbox(
    preds["pred"].cpu().numpy(),
    preds["pred_uct"].cpu().numpy(),
    Y_test.cpu().numpy(),
    X_test.cpu().numpy(),
)